<a href="https://colab.research.google.com/github/uLuum/Firebird_Extraction_Database-Accurate-Desktop/blob/main/Pipeline%20Extract%20Firebird%20Database%20(.gdb).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# UPLOAD DATABASE (.gdb) TO DRIVE (MOUNTING)
# ============================================================

import os
import re
from google.colab import files

# Upload file database (.gdb)
print("Upload the files .gdb:")
uploaded = files.upload()

if not uploaded:
    raise Exception("Nothing uploads, reupload again.")

# Mengambil nama file yang diunggah secara dinamis
db_filename = list(uploaded.keys())[0]
db_path = f"/content/{db_filename}"

print(f"\nUpload Completed: {db_filename}")
print("Go to setup the Dependency & Firebird!\n")

Upload file type .gdb:


Saving Sample.gdb to Sample.gdb

Upload Completed: Sample.gdb
Go to setup the Dependency & Firebird!



In [ ]:
# ============================================================
# SETUP LIBRARY & DEPENDENCY FIREBIRD
# ============================================================

# Checking Ubuntu Version
!lsb_release -cs

# Install libncurses5/libtinfo5 (old version) directly via dpkg (Debian Package)
!wget -q http://archive.ubuntu.com/ubuntu/pool/universe/n/ncurses/libtinfo5_6.3-2ubuntu0.3_amd64.deb
!wget -q http://archive.ubuntu.com/ubuntu/pool/universe/n/ncurses/libncurses5_6.3-2ubuntu0.3_amd64.deb
!dpkg -i libtinfo5_6.3-2ubuntu0.3_amd64.deb libncurses5_6.3-2ubuntu0.3_amd64.deb
!ldconfig

# Download + Extract + Install Firebird 2.5 ODS 11.1 natively, as provided by Accurate Desktop Firebird 2.1
!wget -q https://github.com/FirebirdSQL/firebird/releases/download/R2_5_9/FirebirdCS-2.5.9.27139-0.amd64.tar.gz -O firebird.tar.gz
!tar -xzf firebird.tar.gz
%cd FirebirdCS-2.5.9.27139-0.amd64
!chmod +x install.sh
!./install.sh -silent
%cd /content

# Fix the libfbclient.so.2 linker path issue (ctypes/fdb can't find /usr/lib64 by default) for symlink to connecting the database
!ln -sf /opt/firebird/lib/libfbclient.so.2 /usr/lib/x86_64-linux-gnu/libfbclient.so.2
!ldconfig

# Running the Firebird Server & Copying the database to local server
!/opt/firebird/bin/fbguard -daemon
!sleep 2
!ps aux | grep fb_smp_server

No LSB modules are available.
noble
Selecting previously unselected package libtinfo5:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack libtinfo5_6.3-2ubuntu0.3_amd64.deb ...
Unpacking libtinfo5:amd64 (6.3-2ubuntu0.3) ...
Selecting previously unselected package libncurses5:amd64.
Preparing to unpack libncurses5_6.3-2ubuntu0.3_amd64.deb ...
Unpacking libncurses5:amd64 (6.3-2ubuntu0.3) ...
Setting up libtinfo5:amd64 (6.3-2ubuntu0.3) ...
Setting up libncurses5:amd64 (6.3-2ubuntu0.3) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real

In [ ]:
# ============================================================
# READING THE PASSWORD DATABASES AFTER INSTALLATION
# ============================================================

with open('/opt/firebird/SYSDBA.password') as f:
    content = f.read()
sysdba_pw = re.search(r'ISC_PASSWD=(\S+)', content).group(1)

print(f"\n  --- SYSDBA password: {sysdba_pw} ---")


  --- SYSDBA password: Rp8AEa0Z ---


In [ ]:
# Viewing User Defaults on a Firebird Server

!/opt/firebird/bin/gsec -user SYSDBA -password {sysdba_pw} -display

     user name                    uid   gid admin     full name
------------------------------------------------------------------------------------------------
SYSDBA                              0     0           Sql Server Administrator


In [ ]:
# Create New User to Accessed the Database Server

!/opt/firebird/bin/gsec -user SYSDBA -password {sysdba_pw} -add "CPS#1" -pw admin123 #Max 8 Char to Password

# CPS#1 is superuser default like a SYSDBA, SYSDBA can't be use because is same as one of the SQL role name with administratior Firebird.
# After the added user, You can re-run users display on Firebird Server

In [ ]:
# Test Connection Database Access with New User

!/opt/firebird/bin/isql -user "CPS#1" -password admin123 localhost:/content/{db_filename} #Rename the database

# You can test with query to showing tables or anithing to accessed.
# If the CPS#1 does not have table access, select & added another user by running this query: SELECT DISTINCT RDB$USER FROM RDB$USER_PRIVILEGES;

Database:  localhost:/content/Sample.gdb, User: CPS#1
SQL> quit;


In [ ]:
# ============================================================
# CONNECTING THE FIREBIRD DATABASE WITH PYTHON
# ============================================================

!pip install -q fdb

import fdb
con = fdb.connect(
    dsn=f'localhost:/content/{db_filename}',
    user='CPS#1',
    password='admin123'
)
print("Connected")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.3 MB/s eta 0:00:00
Connected


In [ ]:
# Viewing a Column Structure on the Tables

def show_columns(con, table_name):
    cur = con.cursor()
    cur.execute("""
        SELECT TRIM(RF.RDB$FIELD_NAME), TRIM(F.RDB$FIELD_TYPE)
        FROM RDB$RELATION_FIELDS RF
        JOIN RDB$FIELDS F ON RF.RDB$FIELD_SOURCE = F.RDB$FIELD_NAME
        WHERE RF.RDB$RELATION_NAME = ?
        ORDER BY RF.RDB$FIELD_POSITION
    """, (table_name,))
    for row in cur.fetchall():
        print(row[0])

for t in ['ITEM']: #Header Suspect to Modified the Database
    print(f"\n==== {t} ====")
    try:
        show_columns(con, t)
    except Exception as e:
        print(f"  (skip: {e})")


==== ITEM ====
ITEMNO
ITEMDESCRIPTION
ITEMTYPE
SUBITEM
PARENTITEM
QUANTITY
ONORDER
ONSALES
TAXCODES
UNITPRICE
UNITPRICE2
UNITPRICE3
UNITPRICE4
UNITPRICE5
COST
SUSPENDED
MINIMUMQTY
UNIT1
UNIT2
UNIT3
RATIO2
RATIO3
DISCPC
PREFEREDVENDOR
RESERVED1
RESERVED2
RESERVED3
RESERVED4
RESERVED5
INVENTORYGLACCNT
COGSGLACCNT
PURCHASERETGLACCNT
SALESGLACCNT
SALESRETGLACCNT
NOTES
COSTMETHOD
LOCKED_BY
LOCKED_TIME
PTAXCODES
RESERVED6
RESERVED7
RESERVED8
RESERVED9
RESERVED10
SALESDISCOUNTACCNT
GOODSTRANSITACCNT
FIRSTPARENTITEM
INDENTLEVEL
WAREHOUSEID
PROJECTID
DEPTID
LOGO
FORMAT_LOGO
WEIGHT
DELIVERYLEADTIME
DIMHEIGHT
DIMWIDTH
DIMDEPTH
INVENTORYGROUP
FINISHEDMTRLGLACCNT
CATEGORYID
DEFSTANDARDCOST
TRANSACTIONID
IMPORTEDTRANSACTIONID
BRANCHCODEID
UNITCONTROL
UNBILLEDACCOUNT
QTYCONTROL
LFT
RGT
ISROOT
SERIALNUMBERTYPE
FORCESN
MANAGEEXPIRED
MANAGESN
HSCODE
IMPORTDUTY_RATE
IMPORTDUTY_TYPE
CUKAI_RATE
DELIVERNOSTOCKSN
STOCKOPNAMEDATE
